# 02 — Data Pipeline

Build the final train/val/test splits from the C4_200M Kaggle dataset.

## Steps
1. Load and filter samples from disk
2. Split into train / val / test (95 / 3.3 / 1.7)
3. Save splits as `.tsv` files
4. Verify output

In [1]:
import os

data_dir = "/kaggle/input/datasets/dariocioni/c4200m"
files = sorted(os.listdir(data_dir))
print(f"Files found: {len(files)}")
for f in files:
    size_gb = os.path.getsize(os.path.join(data_dir, f)) / 1e9
    print(f"  {f}  ({size_gb:.2f} GB)")

Files found: 10
  C4_200M.tsv-00000-of-00010  (4.94 GB)
  C4_200M.tsv-00001-of-00010  (4.94 GB)
  C4_200M.tsv-00002-of-00010  (4.94 GB)
  C4_200M.tsv-00003-of-00010  (4.94 GB)
  C4_200M.tsv-00004-of-00010  (4.94 GB)
  C4_200M.tsv-00005-of-00010  (4.94 GB)
  C4_200M.tsv-00006-of-00010  (4.94 GB)
  C4_200M.tsv-00007-of-00010  (4.94 GB)
  C4_200M.tsv-00008-of-00010  (4.93 GB)
  C4_200M.tsv-00009-of-00010  (4.94 GB)


## 1. Setup & Constants

In [2]:
import os
import random
import time

SEED        = 42
TARGET      = 5_000_000
MAX_TOKENS  = 64
MIN_OVERLAP = 0.05
DATA_DIR    = "/kaggle/input/datasets/dariocioni/c4200m"
OUT_DIR     = "/kaggle/working/data"

print("Libraries loaded ✅")
print(f"Target samples : {TARGET:,}")
print(f"Max tokens     : {MAX_TOKENS}")
print(f"Min overlap    : {MIN_OVERLAP}")

Libraries loaded ✅
Target samples : 5,000,000
Max tokens     : 64
Min overlap    : 0.05


## 2. Helper Functions

In [3]:
def token_overlap(s1, s2):
    t1, t2 = set(s1.lower().split()), set(s2.lower().split())
    if not t1 or not t2: return 0.0
    return len(t1 & t2) / max(len(t1), len(t2))

def is_valid(inp, out):
    if not inp or not out: return False
    if len(inp.split()) > MAX_TOKENS or len(out.split()) > MAX_TOKENS: return False
    if inp.strip() == out.strip(): return False
    if token_overlap(inp, out) < MIN_OVERLAP: return False
    return True

print("Helper functions defined ✅")

Helper functions defined ✅


## 3. Load & Filter

In [ ]:
collected = []
seen      = 0
filtered  = 0
start     = time.time()

files = sorted(os.listdir(DATA_DIR))

for fname in files:
    fpath = os.path.join(DATA_DIR, fname)
    print(f"\nReading {fname}...")

    with open(fpath, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) != 2:
                filtered += 1
                seen += 1
                continue

            inp, out = parts[0].strip(), parts[1].strip()
            seen += 1

            if is_valid(inp, out):
                collected.append((inp, out))
            else:
                filtered += 1

            if seen % 500_000 == 0:
                elapsed = time.time() - start
                print(f"  Seen: {seen:>9,} | Collected: {len(collected):>8,} | Filtered: {filtered:>7,} | {elapsed:.0f}s elapsed")

            if len(collected) >= TARGET:
                break

    if len(collected) >= TARGET:
        break

elapsed = time.time() - start
mins, secs = divmod(int(elapsed), 60)
print(f"\nDone in {mins}m {secs}s")
print(f"  Total seen     : {seen:,}")
print(f"  Total collected: {len(collected):,}")
print(f"  Total filtered : {filtered:,} ({100*filtered/seen:.1f}%)")


Reading C4_200M.tsv-00000-of-00010...
  Seen:   500,000 | Collected:  484,394 | Filtered:  15,606 | 9s elapsed
  Seen: 1,000,000 | Collected:  968,923 | Filtered:  31,077 | 18s elapsed
  Seen: 1,500,000 | Collected: 1,453,387 | Filtered:  46,613 | 27s elapsed
  Seen: 2,000,000 | Collected: 1,938,019 | Filtered:  61,981 | 35s elapsed
  Seen: 2,500,000 | Collected: 2,422,733 | Filtered:  77,267 | 44s elapsed
  Seen: 3,000,000 | Collected: 2,907,175 | Filtered:  92,825 | 53s elapsed
  Seen: 3,500,000 | Collected: 3,391,661 | Filtered: 108,339 | 61s elapsed
  Seen: 4,000,000 | Collected: 3,876,238 | Filtered: 123,762 | 70s elapsed
  Seen: 4,500,000 | Collected: 4,360,909 | Filtered: 139,091 | 79s elapsed
  Seen: 5,000,000 | Collected: 4,845,298 | Filtered: 154,702 | 87s elapsed

Done in 1m 30s
  Total seen     : 5,159,785
  Total collected: 5,000,000
  Total filtered : 159,785 (3.1%)


## 4. Split into Train / Val / Test

In [5]:
random.seed(SEED)
random.shuffle(collected)

n       = len(collected)
n_train = int(n * 0.95)
n_val   = int(n * 0.033)

train = collected[:n_train]
val   = collected[n_train : n_train + n_val]
test  = collected[n_train + n_val :]

print(f"Split sizes:")
print(f"  Train : {len(train):,} ({100*len(train)/n:.1f}%)")
print(f"  Val   : {len(val):,}  ({100*len(val)/n:.1f}%)")
print(f"  Test  : {len(test):,}  ({100*len(test)/n:.1f}%)")

Split sizes:
  Train : 4,750,000 (95.0%)
  Val   : 165,000  (3.3%)
  Test  : 85,000  (1.7%)


## 5. Save Splits as TSV

In [6]:
def save_tsv(samples, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for inp, out in samples:
            inp_clean = inp.replace("\t", " ")
            out_clean = out.replace("\t", " ")
            f.write(f"{inp_clean}\t{out_clean}\n")

train_path = os.path.join(OUT_DIR, "train.tsv")
val_path   = os.path.join(OUT_DIR, "val.tsv")
test_path  = os.path.join(OUT_DIR, "test.tsv")

print("Saving train.tsv..."); save_tsv(train, train_path); print(f"  ✅ {len(train):,} rows")
print("Saving val.tsv...");   save_tsv(val,   val_path);   print(f"  ✅ {len(val):,} rows")
print("Saving test.tsv...");  save_tsv(test,  test_path);  print(f"  ✅ {len(test):,} rows")

Saving train.tsv...
  ✅ 4,750,000 rows
Saving val.tsv...
  ✅ 165,000 rows
Saving test.tsv...
  ✅ 85,000 rows


## 6. Sanity Check

In [ ]:
def read_tsv_sample(path, n=3):
    samples = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n: break
            parts = line.strip().split("\t")
            if len(parts) == 2:
                samples.append((parts[0], parts[1]))
    return samples

print("First 3 rows of train.tsv:")
for inp, out in read_tsv_sample(train_path):
    print(f"  IN : {inp[:80]}")
    print(f"  OUT: {out[:80]}")
    print()

for name, path in [("train", train_path), ("val", val_path), ("test", test_path)]:
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {name}.tsv → {size_mb:.1f} MB")

First 3 rows of train.tsv:
  IN : He presented theresults at the american society of Human Genetics 2015 Annual Me
  OUT: He presented the results at the American Society of Human Genetics 2015 Annual M

  IN : Some of the other programe Jesse has produced and directed include s concert vid
  OUT: Some of the other programs Jesse has produced and directed include concert video

  IN : The new 15 percents tax bracket kicks in and applies to incomes above the 0 perc
  OUT: The new 15 percent tax bracket kicks in and applies to incomes above the 0-perce

  train.tsv → 1184.8 MB
  val.tsv → 41.3 MB
  test.tsv → 21.2 MB
